# 04. LoRAファインチューニング：Unslothで効率的な学習

このノートブックでは、**LoRA（Low-Rank Adaptation）** を使って大規模言語モデルを効率的にファインチューニングする方法を学びます。
Unslothライブラリを使うことで、通常の2〜5倍の速度で学習できます。

> **重要：このノートブックはGPU環境（Google Colab推奨）での実行を想定しています**
>
> - Google Colab（無料枠でもT4 GPUで動作確認済み）
> - ローカルのNVIDIA GPU（VRAM 8GB以上推奨）
>
> CPU環境ではコードセルの一部（実際の学習部分）はコメントアウトしており、概念学習・コード確認のみ可能です。
> Colabで開く場合：ランタイム → ランタイムのタイプを変更 → T4 GPU を選択

In [ ]:
# インストール（GPU環境で実行してください）
# Google Colabの場合、以下のコメントを外して実行

# !pip install unsloth
# !pip install xformers trl peft accelerate bitsandbytes
# !pip install datasets transformers

# Unslothの公式ページ: https://github.com/unslothai/unsloth
# Colab向けインストールコマンドは公式READMEを参照

print("インストールコマンドを確認しました")
print("GPU環境では上記のコメントを外して実行してください")

## 1. ファインチューニングとは？LoRAとは？

### ファインチューニングの概念

**事前学習済みモデル（Pre-trained Model）** は大量のテキストで汎用的な知識を習得しています。
しかし特定のタスク（例：カスタマーサポート、医療問答）に特化させたい場合、**ファインチューニング** を行います。

```
事前学習（Pre-training）
┌─────────────────────────────────────────┐
│  インターネット全体のテキスト（数TB）      │
│  → 汎用的な言語理解・生成能力を獲得       │
└─────────────────┬───────────────────────┘
                  │  数週間〜数ヶ月、数千万ドル
                  ▼
         ベースモデル（7B〜70Bパラメータ）
                  │
                  │ ファインチューニング
                  ▼
┌─────────────────────────────────────────┐
│  特定タスクのデータ（数百〜数千サンプル）  │
│  → 特定ドメインの振る舞いを学習           │
└─────────────────┬───────────────────────┘
                  │  数時間〜数日
                  ▼
         特化型モデル（カスタマイズ済み）
```

### フルファインチューニング vs LoRA

**フルファインチューニング（Full Fine-tuning）** の問題点：
- 7Bモデルで約14GB以上のVRAMが必要
- 全パラメータを更新するため時間・コストが大きい
- 元のモデルの汎用性を失いやすい（破滅的忘却）

```
フルファインチューニング
┌──────────────────────────────┐
│  元のモデル（70億パラメータ）  │
│  ████████████████████████   │ ← 全部更新
│  ████████████████████████   │
│  ████████████████████████   │
└──────────────────────────────┘
  VRAM: ~28GB以上、時間: 数時間〜

LoRA（Low-Rank Adaptation）
┌──────────────────────────────┐
│  元のモデル（70億パラメータ）  │
│  ░░░░░░░░░░░░░░░░░░░░░░░░   │ ← フリーズ（更新しない）
│  ░░░░░░░░░░░░░░░░░░░░░░░░   │
│  ░░░░░░░░░░░░░░░░░░░░░░░░   │
└──────────┬──────────────────┘
           │
      ┌────┴────┐
      │ LoRA層  │ ← 小さな追加パラメータだけ更新
      │ A × B   │   （元モデルの0.1〜1%程度）
      └─────────┘
  VRAM: ~6GB〜、時間: 数十分〜
```

### LoRAの数学的な仕組み

通常の重み行列 `W`（大きい）を直接更新する代わりに、
低ランク行列 `A`（小）と `B`（小）の積 `ΔW = A × B` だけを学習します。

```
元の重み: W (d × d) = 4096 × 4096 = 1677万パラメータ

LoRA:     ΔW = A × B
          A: (d × r) = 4096 × 16  = 6.5万パラメータ
          B: (r × d) = 16 × 4096  = 6.5万パラメータ
          合計: 13万パラメータ（元の0.8%！）

推論時: output = (W + α × ΔW) × input
```

**ランク（r）** が小さいほど学習パラメータが少なく高速・省メモリですが、表現力も下がります。
通常 r=8〜64 が使われます。

### QLoRA（Quantized LoRA）

Unslothが使用するQLoRAは、さらに進化した手法です：
- モデルを4bit量子化（通常は16bit/32bit）でロード
- その上にLoRAを適用
- メモリ使用量をさらに削減（7Bモデルが約4GBで動作）

## 2. 環境セットアップ

In [ ]:
# 環境チェック：GPU・必要ライブラリの確認
import sys
import subprocess

# GPU確認
def check_gpu():
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                               capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            gpu_info = result.stdout.strip()
            print(f"GPU検出: {gpu_info}")
            return True
        else:
            print("GPU未検出（CPUモードで実行中）")
            return False
    except Exception:
        print("GPU確認失敗（nvidia-smiが見つかりません）")
        return False

HAS_GPU = check_gpu()

# Unslothの利用可否確認
try:
    from unsloth import FastLanguageModel
    HAS_UNSLOTH = True
    print("Unsloth: インストール済み")
except ImportError:
    HAS_UNSLOTH = False
    print("Unsloth: 未インストール")
    print("  → GPU環境で: pip install unsloth")

# その他のライブラリ確認
libraries = {
    'transformers': 'transformers',
    'datasets': 'datasets',
    'peft': 'peft',
    'trl': 'trl',
    'torch': 'torch'
}

print("\nライブラリ確認:")
for name, module in libraries.items():
    try:
        __import__(module)
        print(f"  {name}: OK")
    except ImportError:
        print(f"  {name}: 未インストール")

print(f"\nPython: {sys.version.split()[0]}")
print(f"\n実行モード: {'GPU（フルトレーニング可能）' if HAS_GPU and HAS_UNSLOTH else 'CPU（コード確認モード）'}")

## 3. モデルの読み込みとLoRA設定

In [ ]:
# モデル設定
# Qwen2.5-7B-Instruct の4bit量子化版（Unsloth最適化済み）
MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

# LoRAのハイパーパラメータ
LORA_CONFIG = {
    "r": 16,              # LoRAランク（低いほど高速・省メモリ、高いほど表現力↑）
    "lora_alpha": 16,     # スケーリング係数（通常はrと同じ値）
    "lora_dropout": 0,    # ドロップアウト（Unslothでは0推奨）
    "bias": "none",       # バイアスの扱い
    "use_gradient_checkpointing": "unsloth",  # メモリ効率化
}

# LoRAを適用するレイヤー（アテンション層とFFN層）
TARGET_MODULES = [
    "q_proj",   # Query projection
    "k_proj",   # Key projection
    "v_proj",   # Value projection
    "o_proj",   # Output projection
    "gate_proj", # FFN gate
    "up_proj",   # FFN up
    "down_proj", # FFN down
]

print(f"使用モデル: {MODEL}")
print(f"\nLoRA設定:")
for k, v in LORA_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nLoRA対象レイヤー: {TARGET_MODULES}")

In [ ]:
# ===== GPU環境でのみ実行 =====
# Unsloth FastLanguageModel でモデルを読み込む

if HAS_UNSLOTH and HAS_GPU:
    from unsloth import FastLanguageModel
    import torch

    # モデルとトークナイザーの読み込み（4bit量子化）
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL,
        max_seq_length=2048,      # 最大シーケンス長
        dtype=None,               # 自動検出（bfloat16推奨）
        load_in_4bit=True,        # 4bit量子化でロード
    )

    # LoRAアダプターを追加
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_CONFIG["r"],
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_CONFIG["lora_alpha"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        use_gradient_checkpointing=LORA_CONFIG["use_gradient_checkpointing"],
        random_state=42,
        use_rslora=False,   # Rank Stabilized LoRA（オフ）
    )

    # パラメータ数の確認
    model.print_trainable_parameters()
    print("\nモデルの読み込みとLoRA設定が完了しました")

else:
    print("[CPU/デモモード] モデル読み込みをスキップ")
    print("実際のコードの内容を確認してください")
    print()
    print("GPU環境での出力例:")
    print("  trainable params: 41,943,040 || all params: 3,784,196,096 || trainable%: 1.1082")
    print("  ※ 全パラメータの約1%だけ学習 → 高速・省メモリ")

## 4. データセットの準備

ファインチューニングには **指示応答形式（Instruction Tuning）** のデータセットが一般的です。

今回は **Alpaca形式** を使います：
```
### 指示:
{instruction}

### 入力:
{input}  ← オプション

### 回答:
{output}
```

In [ ]:
# サンプルデータセット（日本語Q&A）
# 実際は数百〜数千サンプル必要です

sample_data = [
    {
        "instruction": "Pythonで簡単なFizzBuzzを書いてください。",
        "input": "",
        "output": """for i in range(1, 101):
    if i % 15 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)"""
    },
    {
        "instruction": "機械学習とディープラーニングの違いを説明してください。",
        "input": "",
        "output": "機械学習は、データからパターンを学習してタスクを実行するAI技術の総称です。一方、ディープラーニングは機械学習の一手法で、多層のニューラルネットワークを使って特徴を自動的に学習します。ディープラーニングは機械学習の部分集合であり、特に画像認識や自然言語処理で優れた性能を発揮します。"
    },
    {
        "instruction": "以下のテキストを要約してください。",
        "input": "人工知能（AI）は、コンピュータが人間のような知的行動を模倣する技術です。機械学習、自然言語処理、コンピュータビジョンなど多くの分野を含み、医療診断、自動運転、翻訳など様々な応用があります。近年、大規模言語モデル（LLM）の登場により、AIの能力は飛躍的に向上しました。",
        "output": "AIはコンピュータが知的行動を模倣する技術で、機械学習やNLPなど多分野を含む。医療・自動運転・翻訳等に活用され、LLMの登場で能力が大幅に向上した。"
    },
    {
        "instruction": "東京の主要な観光地を3つ挙げて、それぞれの特徴を説明してください。",
        "input": "",
        "output": "1. **浅草・浅草寺**: 東京最古の寺院で、雷門と仲見世通りが有名。江戸の風情が残る下町文化を体験できます。\n2. **新宿**: 世界最大級のターミナル駅を中心に、歌舞伎町の繁華街から新宿御苑まで多様な魅力を持つエリア。\n3. **渋谷スクランブル交差点**: 世界で最も有名な交差点の一つ。周辺は若者文化の発信地として知られています。"
    },
    {
        "instruction": "再帰関数を使ってフィボナッチ数列を計算するPython関数を書いてください。",
        "input": "",
        "output": """def fibonacci(n):
    \"\"\"フィボナッチ数列のn番目の値を返す（0始まり）\"\"\"
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

# テスト
for i in range(10):
    print(f"fibonacci({i}) = {fibonacci(i)}")"""
    }
]

print(f"サンプルデータ数: {len(sample_data)}件")
print("\n最初のサンプル:")
print(f"  instruction: {sample_data[0]['instruction'][:50]}...")
print(f"  output: {sample_data[0]['output'][:50]}...")

In [ ]:
# Alpacaプロンプトテンプレートとデータのフォーマット

ALPACA_PROMPT = """以下は、タスクを説明する指示と、追加のコンテキストを提供する入力の組み合わせです。
要求を適切に満たす回答を書いてください。

### 指示:
{instruction}

### 入力:
{input}

### 回答:
{output}"""

def format_alpaca(example, eos_token="<|endoftext|>"):
    """Alpaca形式にデータをフォーマットする"""
    text = ALPACA_PROMPT.format(
        instruction=example["instruction"],
        input=example["input"],
        output=example["output"]
    ) + eos_token  # 終端トークンを追加
    return {"text": text}

# フォーマットの確認
formatted = format_alpaca(sample_data[1])
print("フォーマット済みサンプル:")
print("=" * 60)
print(formatted["text"][:500])
print("...")

# データセットオブジェクトの作成
try:
    from datasets import Dataset
    dataset = Dataset.from_list(sample_data)
    dataset = dataset.map(format_alpaca)
    print(f"\nデータセット作成完了: {len(dataset)}件")
    print(f"カラム: {dataset.column_names}")
except ImportError:
    print("\n[datasets未インストール] データセット作成をスキップ")
    print("pip install datasets でインストールしてください")

## 5. 学習の実行

TRL（Transformer Reinforcement Learning）ライブラリの `SFTTrainer` を使います。
SFT = Supervised Fine-Tuning（教師あり学習によるファインチューニング）

In [ ]:
# SFTTrainerの設定（GPU環境のみ実際に動作）

TRAINING_CONFIG = {
    # 基本設定
    "per_device_train_batch_size": 2,    # バッチサイズ（VRAMに応じて調整）
    "gradient_accumulation_steps": 4,    # 勾配累積（実効バッチサイズ = 2×4 = 8）
    "max_steps": 60,                     # 最大ステップ数（小さいデータセット用）
    # "num_train_epochs": 3,             # エポック数（max_stepsの代わりに使用可）

    # 最適化
    "learning_rate": 2e-4,               # 学習率
    "warmup_steps": 5,                   # ウォームアップステップ
    "lr_scheduler_type": "linear",       # 学習率スケジューラ
    "optim": "adamw_8bit",              # 8bit AdamW（省メモリ）
    "weight_decay": 0.01,               # 重み減衰

    # 出力
    "output_dir": "./finetuned_model",
    "save_steps": 20,                    # 20ステップごとに保存
    "logging_steps": 5,                  # 5ステップごとにログ出力

    # 効率化
    "fp16": False,                       # T4 GPUの場合True
    "bf16": True,                        # A100/H100の場合True（Ampere以降）
}

print("学習設定:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")

print("\n実効バッチサイズ:", 
      TRAINING_CONFIG["per_device_train_batch_size"] * TRAINING_CONFIG["gradient_accumulation_steps"])

In [ ]:
# ===== GPU環境でのみ実行 =====
# SFTTrainerで学習を実行

if HAS_UNSLOTH and HAS_GPU:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from unsloth import is_bfloat16_supported

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=2048,
        dataset_num_proc=2,
        packing=False,  # Trueにすると短いシーケンスをパック→高速化
        args=TrainingArguments(
            per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
            gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
            max_steps=TRAINING_CONFIG["max_steps"],
            learning_rate=TRAINING_CONFIG["learning_rate"],
            warmup_steps=TRAINING_CONFIG["warmup_steps"],
            lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
            optim=TRAINING_CONFIG["optim"],
            weight_decay=TRAINING_CONFIG["weight_decay"],
            output_dir=TRAINING_CONFIG["output_dir"],
            logging_steps=TRAINING_CONFIG["logging_steps"],
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            seed=42,
        ),
    )

    # VRAM使用量を表示
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU: {gpu_stats.name}")
    print(f"VRAM使用量（学習前）: {start_gpu_memory}GB / {max_memory}GB")

    # 学習開始
    print("\n学習を開始します...")
    trainer_stats = trainer.train()

    # 学習後の統計
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    print(f"\n学習完了!")
    print(f"学習時間: {trainer_stats.metrics['train_runtime']:.0f}秒")
    print(f"最終Loss: {trainer_stats.metrics['train_loss']:.4f}")
    print(f"VRAM使用量（学習後）: {used_memory}GB / {max_memory}GB")

else:
    print("[CPU/デモモード] 学習をスキップ")
    print("\nGPU環境での出力例:")
    print("  GPU: Tesla T4")
    print("  VRAM使用量（学習前）: 5.2GB / 15.9GB")
    print("  {'loss': 1.2345, 'learning_rate': 0.0001, 'epoch': 0.5}")
    print("  学習完了!")
    print("  学習時間: 312秒")
    print("  最終Loss: 0.8234")

## 6. モデルの評価と保存

In [ ]:
# ===== GPU環境でのみ実行 =====
# ファインチューニング後の推論テスト

def test_finetuned_model(model, tokenizer, question):
    """ファインチューニング済みモデルでテスト推論"""
    # 推論モードに切り替え（高速化）
    FastLanguageModel.for_inference(model)

    # プロンプトの作成（回答部分は空）
    prompt = ALPACA_PROMPT.format(
        instruction=question,
        input="",
        output=""  # 空のまま → モデルが生成
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to("cuda")

    # 生成
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
    )

    # デコード（入力部分を除いて回答のみ取得）
    input_length = inputs["input_ids"].shape[1]
    generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return generated


if HAS_UNSLOTH and HAS_GPU:
    test_question = "Pythonでリストの重複要素を削除する方法を教えてください。"
    print(f"質問: {test_question}")
    print("\n回答:")
    answer = test_finetuned_model(model, tokenizer, test_question)
    print(answer)
else:
    print("[CPU/デモモード] 推論テストをスキップ")
    print("\n期待される回答例:")
    print("質問: Pythonでリストの重複要素を削除する方法を教えてください。")
    print("回答: リストの重複を削除するには、set()を使う方法が最も簡単です。")
    print("  my_list = [1, 2, 2, 3, 3, 3]")
    print("  unique_list = list(set(my_list))")

In [ ]:
# ===== GPU環境でのみ実行 =====
# モデルの保存（Hugging Face形式 / GGUF形式）

SAVE_PATH = "./finetuned_qwen2_5"

if HAS_UNSLOTH and HAS_GPU:
    # 1. LoRAアダプターのみ保存（軽量、再学習用）
    model.save_pretrained(SAVE_PATH + "_lora")
    tokenizer.save_pretrained(SAVE_PATH + "_lora")
    print(f"LoRAアダプター保存完了: {SAVE_PATH}_lora")

    # 2. マージしたモデルを保存（デプロイ用）
    model.save_pretrained_merged(
        SAVE_PATH + "_merged",
        tokenizer,
        save_method="merged_16bit"  # 16bitで保存
    )
    print(f"マージ済みモデル保存完了: {SAVE_PATH}_merged")

    # 3. GGUF形式で保存（Ollama/llama.cpp用）
    # Ollamaで動かすために必要な形式
    model.save_pretrained_gguf(
        SAVE_PATH + "_gguf",
        tokenizer,
        quantization_method="q4_k_m"  # 4bit量子化
    )
    print(f"GGUF形式保存完了: {SAVE_PATH}_gguf")

    # 4. Ollamaにインポート（Modelfileが必要）
    modelfile_content = f"""FROM {SAVE_PATH}_gguf/unsloth.Q4_K_M.gguf
PARAMETER temperature 0.7
PARAMETER stop "<|endoftext|>"
SYSTEM あなたは親切なアシスタントです。日本語で回答してください。
"""
    with open("Modelfile", "w") as f:
        f.write(modelfile_content)

    print("\nOllamaへのインポート方法:")
    print("  ollama create my-finetuned-model -f Modelfile")
    print("  ollama run my-finetuned-model")

else:
    print("[CPU/デモモード] 保存をスキップ")
    print("\n保存オプション:")
    print("  1. LoRAアダプターのみ（~100MB）: 再学習・継続学習用")
    print("  2. マージ済みモデル（~14GB）: デプロイ用フル精度")
    print("  3. GGUF Q4_K_M（~4GB）: Ollama/llama.cpp用")
    print("\nOllamaへのインポート手順:")
    print("  1. GGUFファイルを作成")
    print("  2. Modelfileを作成")
    print("  3. ollama create my-model -f Modelfile")
    print("  4. ollama run my-model")

## まとめ：LoRAのメリットとデメリット

### メリット

| 項目 | フルFT | LoRA | 改善率 |
|------|--------|------|--------|
| VRAM使用量 | ~28GB | ~6GB | 約4.5倍削減 |
| 学習時間 | 数時間 | 数十分 | 約5倍高速 |
| ストレージ | ~14GB | ~100MB | 約140倍削減 |
| 破滅的忘却 | 起きやすい | 起きにくい | 改善 |

### デメリットと注意点

1. **データ品質が最重要**: 少量でも高品質なデータが必要。ゴミデータを入れるとモデルが劣化
2. **ランクの選択**: r が小さすぎると学習不足、大きすぎるとフルFTと変わらなくなる
3. **過学習のリスク**: 小さいデータセットでエポック数を増やしすぎると過学習
4. **ベースモデルへの依存**: ベースモデルが持っていない知識は学習できない
5. **ライセンスの確認**: 商用利用する場合はモデルのライセンスを必ず確認

### 次のステップ

- **DPO/RLHF**: 人間のフィードバックを使った強化学習でさらに品質向上
- **データ収集**: 自分のユースケースに合ったデータを集める（最低200件〜）
- **Hugging Face Hub**: ファインチューニング済みモデルを共有・公開
- **評価指標の設定**: 05_llm_evaluation.ipynb で自動評価を学ぶ

### 参考リンク

- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [LoRA論文 (Hu et al., 2021)](https://arxiv.org/abs/2106.09685)
- [QLoRA論文 (Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314)
- [Alpacaデータセット形式](https://github.com/tatsu-lab/stanford_alpaca)